## LightGBM

Main model. Same data preparation and train/test split as `02_baseline`, so the test set is identical.
Tree models handle missing values and outliers natively: no imputation, clipping or scaling.
Model selection uses 5-fold stratified cross-validation on train only. The test set is used once, for the final comparison.

**1 - Install LightGBM**

In [0]:
%pip install lightgbm
dbutils.library.restartPython()

**2 - Imports and constants**

In [0]:
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score

PATH = "/Volumes/workspace/risco_credito/raw/cs-training.csv"
TARGET = "SeriousDlqin2yrs"
PAST_DUE = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate",
]
RANDOM_STATE = 42
N_SPLITS = 5

# Test-set results from 02_baseline (logistic regression)
BASELINE = {"roc_auc": 0.8601, "pr_auc": 0.3977}

**3 - Read CSV, mark missing values and split**

In [0]:
df = pd.read_csv(PATH, index_col=0)

# Flags first (row-level rules, no statistics -> safe before split)
df["past_due_sentinel"] = df[PAST_DUE].isin([96, 98]).any(axis=1).astype(int)
df["income_missing"] = df["MonthlyIncome"].isna().astype(int)
df["dependents_missing"] = df["NumberOfDependents"].isna().astype(int)

# Invalid values -> NaN (kept as NaN: LightGBM handles missing values natively)
df.loc[df["age"] == 0, "age"] = np.nan
df[PAST_DUE] = df[PAST_DUE].where(~df[PAST_DUE].isin([96, 98]))

X = df.drop(columns=TARGET)
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(f"train: {X_train.shape} | default rate {y_train.mean():.4f}")
print(f"test:  {X_test.shape} | default rate {y_test.mean():.4f}")

**4 - Cross-validate LightGBM on train**

In [0]:
from sklearn.model_selection import cross_validate

lgbm = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,          # default tree size
    random_state=RANDOM_STATE,
    verbose=-1,             # silence training logs
)

cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

scores = cross_validate(
    lgbm, X_train, y_train, cv=cv,
    scoring={"roc_auc": "roc_auc", "pr_auc": "average_precision"},
)

for m in ["roc_auc", "pr_auc"]:
    s = scores[f"test_{m}"]  # "test_" = validation fold, not our held-out test set
    print(f"{m}: {s.mean():.4f} ± {s.std():.4f} | folds {np.round(s, 4)} | baseline (test) {BASELINE[m]:.4f}")